
# 00E — UNSW-NB15 Fingerprint-Multiplicity Feasibility Audit

**Experiment 1 v2 — structural diagnostic only**

00D established a monotone relationship between pure-benign fingerprint-group size and false-positive rate. This notebook audits whether the corrected benign allocation pool can support a **jointly source-aware and fingerprint-multiplicity-aware atomic partition**.

It does **not** modify PRE-01 or PRE-02, retrain a model, access attack scores/outcomes, access final-test outcomes, or authorize formal Notebook 01.

The multiplicity strata are frozen prospectively as:

\[
1,\;2,\;3\!-\!4,\;5\!-\!9,\;10\!-\!19,\;20\!-\!49,\;50\!-\!99,\;\ge100.
\]

The notebook reports both **rows** and **groups**, cross-tabulates `source × multiplicity`, and computes an exact optimistic multiplicity-only lower bound on total-variation (TV) distance for the tightest pure-benign role under whole-group atomicity.


In [1]:

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import hashlib, json, os
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

DATASET_NAME = "UNSW-NB15"
PROTOCOL_VERSION = "Experiment_1_v2"
MODEL_BRANCH_VERSION = "v2_source_stratified_20260918"
PRE01_DIRNAME = "partitioning_s8_s11_v2_source_stratified_seed20260914"
OUTPUT_STAGE = "preformal_fingerprint_multiplicity_feasibility_audit"

PROJECT_FOLDER_NAME = "EXPERIMENT1_V2"
PRIMARY_PROJECT = Path("/content/drive/MyDrive") / PROJECT_FOLDER_NAME
FALLBACK_PROJECT = Path("/content/exp1v2_google_drive/MyDrive") / PROJECT_FOLDER_NAME
PROJECT_ROOT_OVERRIDE = None

ROLE_ORDER = [
    "benign_weight_train",
    "benign_early_stop",
    "benign_threshold_calibration",
    "development",
    "final_test",
]
SIZE_BAND_ORDER = ["1","2","3-4","5-9","10-19","20-49","50-99",">=100"]

def require(cond, msg):
    if not bool(cond):
        print("\nEXPERIMENT 1 v2 STRUCTURAL DIAGNOSTIC: FAIL")
        print("No repartitioning or formal modeling is authorized.")
        raise RuntimeError(str(msg))

def sha256_file(path, chunk_size=1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        while True:
            b=f.read(chunk_size)
            if not b: break
            h.update(b)
    return h.hexdigest()

def canonical_json_bytes(obj):
    return json.dumps(obj, sort_keys=True, separators=(",",":"), ensure_ascii=False).encode("utf-8")

def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()

def atomic_write_csv(path, df):
    path=Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp=path.with_suffix(path.suffix+".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)

def atomic_write_json(path, obj):
    path=Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp=path.with_suffix(path.suffix+".tmp")
    with open(tmp,"w",encoding="utf-8") as f:
        json.dump(obj,f,indent=2,ensure_ascii=False)
    os.replace(tmp,path)

def size_band(n):
    n=int(n)
    if n==1: return "1"
    if n==2: return "2"
    if n<=4: return "3-4"
    if n<=9: return "5-9"
    if n<=19: return "10-19"
    if n<=49: return "20-49"
    if n<=99: return "50-99"
    return ">=100"

print("Dataset:", DATASET_NAME)
print("Stage:", OUTPUT_STAGE)
print("Formal Notebook 01 remains BLOCKED.")


Dataset: UNSW-NB15
Stage: preformal_fingerprint_multiplicity_feasibility_audit
Formal Notebook 01 remains BLOCKED.


In [2]:

def resolve_project_root():
    if PROJECT_ROOT_OVERRIDE is not None:
        p=Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
        require(p.exists(), f"PROJECT_ROOT_OVERRIDE does not exist: {p}")
        return p
    if PRIMARY_PROJECT.exists():
        return PRIMARY_PROJECT.resolve()
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as exc:
        print("Drive mount attempt:", repr(exc))
    if PRIMARY_PROJECT.exists():
        return PRIMARY_PROJECT.resolve()
    if FALLBACK_PROJECT.exists():
        return FALLBACK_PROJECT.resolve()
    try:
        from google.colab import drive
        drive.mount("/content/exp1v2_google_drive", force_remount=False)
    except Exception as exc:
        print("Fallback mount attempt:", repr(exc))
    require(FALLBACK_PROJECT.exists(), "Could not locate EXPERIMENT1_V2.")
    return FALLBACK_PROJECT.resolve()

PROJECT_ROOT=resolve_project_root()
DATASET_ROOT=PROJECT_ROOT/"DATASETS"/DATASET_NAME
PRE01_ROOT=DATASET_ROOT/PRE01_DIRNAME
MEMBERSHIP_PATH=PRE01_ROOT/"partition_membership.parquet"
OUTPUT_ROOT=DATASET_ROOT/"modelling"/OUTPUT_STAGE/MODEL_BRANCH_VERSION
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

require(MEMBERSHIP_PATH.exists(), f"Missing membership: {MEMBERSHIP_PATH}")
MEMBERSHIP_SHA256=sha256_file(MEMBERSHIP_PATH)
schema_names=pq.read_schema(MEMBERSHIP_PATH).names
required={"row_id","feature_fingerprint","fingerprint_group_class","partition"}
require(required.issubset(set(schema_names)), f"Missing columns: {sorted(required-set(schema_names))}")

SOURCE_CANDIDATES=["source_file","source_csv","source_filename","source_csv_name","source_name","dataset_source","source"]
source_col=next((c for c in SOURCE_CANDIDATES if c in schema_names), None)
require(source_col is not None, f"No source column discovered. Available: {schema_names}")

print("Membership SHA-256:", MEMBERSHIP_SHA256)
print("Source column:", source_col)


Mounted at /content/drive
Membership SHA-256: 3114e887aeff514be147d68a9f938d9c889d2c89a21a1ccffd6ff65fcb4a4fac
Source column: source_file


In [3]:

cols=["row_id","feature_fingerprint","fingerprint_group_class","partition",source_col]
df=pd.read_parquet(MEMBERSHIP_PATH, columns=cols, engine="pyarrow")

observed_classes=sorted(df["fingerprint_group_class"].astype(str).unique().tolist())
pure_candidates=[x for x in observed_classes if ("pure" in x.lower() and "benign" in x.lower() and "mixed" not in x.lower())]
require(len(pure_candidates)==1, f"Expected one pure-benign class; observed={observed_classes}")
PURE_BENIGN_CLASS=pure_candidates[0]

pure=df.loc[df["fingerprint_group_class"].astype(str).eq(PURE_BENIGN_CLASS)].copy()
require(len(pure)>0, "No pure-benign rows found.")
require(set(pure["partition"].unique()).issubset(set(ROLE_ORDER)),
        f"Unexpected pure-benign roles: {sorted(set(pure['partition'])-set(ROLE_ORDER))}")

role_nunique=pure.groupby("feature_fingerprint",sort=False)["partition"].nunique()
require(int(role_nunique.max())==1, "A pure-benign fingerprint crosses roles.")

group=(pure.groupby("feature_fingerprint",sort=False)
          .agg(group_size=("row_id","size"), partition=("partition","first"))
          .reset_index())
group["size_band"]=pd.Categorical(group["group_size"].map(size_band),
                                  categories=SIZE_BAND_ORDER, ordered=True)

source_counts=(pure.groupby(["feature_fingerprint",source_col],sort=False)
                   .size().rename("rows_in_source").reset_index())
source_touch=(source_counts.groupby("feature_fingerprint",sort=False)[source_col]
                          .nunique().rename("n_sources_touched").reset_index())
group=group.merge(source_touch,on="feature_fingerprint",how="left",validate="one_to_one")

print("Pure-benign rows:", f"{len(pure):,}")
print("Pure-benign fingerprints:", f"{len(group):,}")
print("Max group size:", int(group["group_size"].max()))
print("Groups touching >1 source:", int((group["n_sources_touched"]>1).sum()))


Pure-benign rows: 2,217,778
Pure-benign fingerprints: 1,938,413
Max group size: 760
Groups touching >1 source: 5266


In [4]:

global_band=(group.groupby("size_band",observed=False)
                  .agg(groups=("feature_fingerprint","size"),
                       rows=("group_size","sum"),
                       mean_group_size=("group_size","mean"),
                       max_group_size=("group_size","max"))
                  .reset_index())
global_band["group_share"]=global_band["groups"]/global_band["groups"].sum()
global_band["row_share"]=global_band["rows"]/global_band["rows"].sum()

role_band=(group.groupby(["partition","size_band"],observed=False)
                .agg(groups=("feature_fingerprint","size"),
                     rows=("group_size","sum"))
                .reset_index())
role_totals=(group.groupby("partition",sort=False)
                  .agg(pure_benign_groups=("feature_fingerprint","size"),
                       pure_benign_rows=("group_size","sum"))
                  .reset_index())
role_band=role_band.merge(role_totals,on="partition",how="left")
role_band["group_share_within_role"]=role_band["groups"]/role_band["pure_benign_groups"]
role_band["row_share_within_role"]=role_band["rows"]/role_band["pure_benign_rows"]
global_share_map=global_band.set_index("size_band")["row_share"].to_dict()
role_band["global_row_share"]=role_band["size_band"].map(global_share_map).astype(float)
role_band["row_share_deviation"]=role_band["row_share_within_role"]-role_band["global_row_share"]

MULTIPLICITY_TV=(role_band.assign(abs_dev=lambda d:d["row_share_deviation"].abs())
                          .groupby("partition",sort=False)
                          .agg(multiplicity_tv_distance=("abs_dev",lambda s:0.5*float(s.sum())),
                               max_abs_band_row_share_deviation=("row_share_deviation",
                                                                 lambda s:float(np.max(np.abs(s)))))
                          .reset_index()
                          .merge(role_totals,on="partition",how="left"))

atomic_write_csv(OUTPUT_ROOT/"global_pure_benign_multiplicity_distribution.csv",global_band)
atomic_write_csv(OUTPUT_ROOT/"pure_benign_multiplicity_by_role.csv",role_band)
atomic_write_csv(OUTPUT_ROOT/"current_role_multiplicity_tv.csv",MULTIPLICITY_TV)

display(global_band)
display(MULTIPLICITY_TV.sort_values("multiplicity_tv_distance",ascending=False))


,size_band,groups,rows,mean_group_size,max_group_size,group_share,row_share
0,1,1882245,1882245,1.000000,1,0.971024,0.848708
1,2,34083,68166,2.000000,2,0.017583,0.030736
2,3-4,11984,40682,3.394693,4,0.006182,0.018344
3,5-9,3772,24579,6.516172,9,0.001946,0.011083
4,10-19,3295,46024,13.967830,19,0.001700,0.020752
5,20-49,2131,62195,29.185828,49,0.001099,0.028044
6,50-99,627,42408,67.636364,99,0.000323,0.019122
7,>=100,276,51479,186.518116,760,0.000142,0.023212


,partition,multiplicity_tv_distance,max_abs_band_row_share_deviation,pure_benign_groups,pure_benign_rows
3,development,0.709426,0.709426,60066,220890
0,benign_early_stop,0.391849,0.373775,120113,221878
2,benign_weight_train,0.151292,0.151292,1109381,1109381
1,benign_threshold_calibration,0.112169,0.112169,325360,332815
4,final_test,0.104568,0.103557,323493,332814


In [5]:

source_counts=source_counts.merge(
    group[["feature_fingerprint","partition","group_size","size_band","n_sources_touched"]],
    on="feature_fingerprint",how="left",validate="many_to_one")

global_source_size=(source_counts.groupby([source_col,"size_band"],observed=False)
                    .agg(rows=("rows_in_source","sum"),
                         groups_touching_cell=("feature_fingerprint","nunique"),
                         max_group_size_touching_cell=("group_size","max"))
                    .reset_index())
global_source_size["row_share_of_pure_pool"]=global_source_size["rows"]/global_source_size["rows"].sum()

role_source_size=(source_counts.groupby(["partition",source_col,"size_band"],observed=False)
                  .agg(rows=("rows_in_source","sum"),
                       groups_touching_cell=("feature_fingerprint","nunique"),
                       max_group_size_touching_cell=("group_size","max"))
                  .reset_index()
                  .merge(role_totals[["partition","pure_benign_rows"]],on="partition",how="left"))
role_source_size["row_share_within_role"]=role_source_size["rows"]/role_source_size["pure_benign_rows"]

gmap=global_source_size.set_index([source_col,"size_band"])["row_share_of_pure_pool"].to_dict()
role_source_size["global_cell_row_share"]=[gmap[(s,b)] for s,b in zip(role_source_size[source_col],role_source_size["size_band"])]
role_source_size["cell_row_share_deviation"]=role_source_size["row_share_within_role"]-role_source_size["global_cell_row_share"]

SOURCE_SIZE_TV=(role_source_size.assign(abs_dev=lambda d:d["cell_row_share_deviation"].abs())
                .groupby("partition",sort=False)
                .agg(source_x_size_tv_distance=("abs_dev",lambda s:0.5*float(s.sum())),
                     max_abs_source_x_size_cell_deviation=("cell_row_share_deviation",
                                                           lambda s:float(np.max(np.abs(s)))))
                .reset_index()
                .merge(role_totals,on="partition",how="left"))

SPARSE_CELLS=global_source_size.copy()
for k in [5,10,25]:
    SPARSE_CELLS[f"sparse_lt_{k}_groups"]=SPARSE_CELLS["groups_touching_cell"]<k

atomic_write_csv(OUTPUT_ROOT/"global_source_x_multiplicity_cells.csv",global_source_size)
atomic_write_csv(OUTPUT_ROOT/"current_role_source_x_multiplicity_cells.csv",role_source_size)
atomic_write_csv(OUTPUT_ROOT/"current_role_source_x_multiplicity_tv.csv",SOURCE_SIZE_TV)
atomic_write_csv(OUTPUT_ROOT/"source_x_multiplicity_sparse_cells.csv",SPARSE_CELLS)

display(SOURCE_SIZE_TV.sort_values("source_x_size_tv_distance",ascending=False))
display(SPARSE_CELLS.loc[SPARSE_CELLS["sparse_lt_25_groups"]]
        .sort_values(["groups_touching_cell",source_col,"size_band"]))


,partition,source_x_size_tv_distance,max_abs_source_x_size_cell_deviation,pure_benign_groups,pure_benign_rows
3,development,0.741892,0.267622,60066,220890
0,benign_early_stop,0.407937,0.188444,120113,221878
2,benign_weight_train,0.151292,0.056096,1109381,1109381
1,benign_threshold_calibration,0.139980,0.037857,325360,332815
4,final_test,0.134073,0.045134,323493,332814


,source_file,size_band,rows,groups_touching_cell,max_group_size_touching_cell,row_share_of_pure_pool,sparse_lt_5_groups,sparse_lt_10_groups,sparse_lt_25_groups
7,UNSW-NB15_1.csv,>=100,571,5,198,0.000257,False,True,True
6,UNSW-NB15_1.csv,50-99,910,15,87,0.000410,False,False,True


In [6]:

role_target_map=dict(zip(role_totals["partition"],role_totals["pure_benign_rows"]))
tightest_role=min(role_target_map,key=role_target_map.get)
tight_target_rows=int(role_target_map[tightest_role])
total_pure_rows=int(group["group_size"].sum())
tight_fraction=tight_target_rows/total_pure_rows

def bounded_subset_sum_bitset(values):
    counts=Counter(map(int,values))
    bits=1
    for value,count in sorted(counts.items()):
        remaining=int(count); power=1
        while remaining>0:
            take=min(power,remaining)
            bits |= bits << (value*take)
            remaining -= take
            power <<= 1
    return bits

def nearest_reachable_sum(bits,target):
    target=float(target)
    lo_t=max(0,int(np.floor(target)))
    hi_t=max(0,int(np.ceil(target)))
    lower_masked=bits & ((1<<(lo_t+1))-1)
    lower=(lower_masked.bit_length()-1) if lower_masked else None
    shifted=bits>>hi_t
    higher=None
    if shifted:
        lowbit=shifted & -shifted
        higher=hi_t+(lowbit.bit_length()-1)
    candidates=[x for x in [lower,higher] if x is not None]
    require(candidates,"No reachable subset sum.")
    return min(candidates,key=lambda x:(abs(x-target),x))

rows=[]; sum_abs_dev=0.0; sum_chosen=0
for band in SIZE_BAND_ORDER:
    vals=group.loc[group["size_band"].astype(str).eq(band),"group_size"].astype(int).tolist()
    total=int(sum(vals)); target=tight_fraction*total
    bits=bounded_subset_sum_bitset(vals)
    nearest=int(nearest_reachable_sum(bits,target))
    abs_dev=abs(nearest-target)
    sum_abs_dev += abs_dev; sum_chosen += nearest
    rows.append({
        "tightest_role":tightest_role,
        "size_band":band,
        "global_band_rows":total,
        "global_band_groups":len(vals),
        "role_fraction_of_pure_pool":tight_fraction,
        "ideal_role_rows_in_band":target,
        "nearest_atomically_reachable_rows_in_band":nearest,
        "absolute_row_deviation_lower_bound_component":abs_dev,
        "largest_group_in_band":max(vals) if vals else 0,
        "mean_group_size_in_band":float(np.mean(vals)) if vals else np.nan,
    })

LOWER=pd.DataFrame(rows)
tv_lower_bound=0.5*sum_abs_dev/tight_target_rows
row_mismatch=int(sum_chosen-tight_target_rows)
LOWER_SUMMARY=pd.DataFrame([{
    "tightest_role":tightest_role,
    "tightest_role_current_pure_benign_rows":tight_target_rows,
    "total_pure_benign_rows":total_pure_rows,
    "tightest_role_fraction_of_pure_pool":tight_fraction,
    "multiplicity_only_independent_bin_tv_lower_bound":tv_lower_bound,
    "independent_bin_nearest_sums_total_rows":int(sum_chosen),
    "independent_bin_total_row_mismatch":row_mismatch,
    "interpretation":"Optimistic exact lower bound for multiplicity balance only; source constraints and joint role-capacity coupling can only make the full problem equally hard or harder."
}])

atomic_write_csv(OUTPUT_ROOT/"tightest_role_multiplicity_atomic_lower_bound_by_band.csv",LOWER)
atomic_write_csv(OUTPUT_ROOT/"tightest_role_multiplicity_atomic_lower_bound_summary.csv",LOWER_SUMMARY)

display(LOWER)
display(LOWER_SUMMARY)


,tightest_role,size_band,global_band_rows,global_band_groups,role_fraction_of_pure_pool,ideal_role_rows_in_band,nearest_atomically_reachable_rows_in_band,absolute_row_deviation_lower_bound_component,largest_group_in_band,mean_group_size_in_band
0,development,1,1882245,1882245,0.0996,187471.017410,187471,0.017410,1,1.000000
1,development,2,68166,34083,0.0996,6789.312429,6790,0.687571,2,2.000000
2,development,3-4,40682,11984,0.0996,4051.914565,4052,0.085435,4,3.394693
3,development,5-9,24579,3772,0.0996,2448.060766,2448,0.060766,9,6.516172
4,development,10-19,46024,3295,0.0996,4583.976106,4584,0.023894,19,13.967830
5,development,20-49,62195,2131,0.0996,6194.602683,6195,0.397317,49,29.185828
6,development,50-99,42408,627,0.0996,4223.823629,4224,0.176371,99,67.636364
7,development,>=100,51479,276,0.0996,5127.292412,5127,0.292412,760,186.518116


,tightest_role,tightest_role_current_pure_benign_rows,total_pure_benign_rows,tightest_role_fraction_of_pure_pool,multiplicity_only_independent_bin_tv_lower_bound,independent_bin_nearest_sums_total_rows,independent_bin_total_row_mismatch,interpretation
0,development,220890,2217778,0.0996,0.000004,220891,1,Optimistic exact lower bound for multiplicity ...


In [7]:

large_bands=["20-49","50-99",">=100"]
LARGE=global_source_size.loc[global_source_size["size_band"].astype(str).isin(large_bands)].copy()
LARGE["tightest_role_ideal_rows"]=LARGE["row_share_of_pure_pool"]*tight_target_rows
LARGE["largest_group_to_ideal_cell_target_ratio"]=(
    LARGE["max_group_size_touching_cell"]/
    LARGE["tightest_role_ideal_rows"].replace(0,np.nan)
)
LARGE["few_groups_for_10pct_like_role"]=LARGE["groups_touching_cell"]<10
atomic_write_csv(OUTPUT_ROOT/"large_band_source_cell_feasibility.csv",LARGE)
display(LARGE.sort_values(["size_band","groups_touching_cell",source_col]))


,source_file,size_band,rows,groups_touching_cell,max_group_size_touching_cell,row_share_of_pure_pool,tightest_role_ideal_rows,largest_group_to_ideal_cell_target_ratio,few_groups_for_10pct_like_role
5,UNSW-NB15_1.csv,20-49,2917,118,48,0.001315,290.532294,0.165214,False
13,UNSW-NB15_2.csv,20-49,7484,391,49,0.003375,745.404076,0.065736,False
29,UNSW-NB15_4.csv,20-49,16166,755,49,0.007289,1610.128579,0.030432,False
21,UNSW-NB15_3.csv,20-49,35628,1489,49,0.016065,3548.537735,0.013809,False
6,UNSW-NB15_1.csv,50-99,910,15,87,0.000410,90.635717,0.959886,False
14,UNSW-NB15_2.csv,50-99,4903,138,99,0.002211,488.337277,0.202729,False
30,UNSW-NB15_4.csv,50-99,14409,346,99,0.006497,1435.131925,0.068983,False
22,UNSW-NB15_3.csv,50-99,22186,483,99,0.010004,2209.718709,0.044802,False
7,UNSW-NB15_1.csv,>=100,571,5,198,0.000257,56.871423,3.481538,True
15,UNSW-NB15_2.csv,>=100,5931,93,760,0.002674,590.725758,1.286553,False



## Interpretation and stopping rule

00E is a **feasibility audit**, not a repartitioner.

- If the multiplicity-only lower bound is near zero and the large-band source cells have enough independent groups, then a source-aware + multiplicity-aware atomic allocator is feasible in principle.
- If large-band cells are sparse, freeze the tolerance against the demonstrated arithmetic before allocation. In those sparse large bands, **multiplicity balance should take priority over source balance**, because multiplicity is now the demonstrated driver of the calibration failure.
- If a prospectively frozen source-aware + multiplicity-aware PRE-01 v3 still fails the benign-only quantile-transfer smoke gate, **do not amend PRE-01 again**. Treat the inability to obtain an exchangeable benign calibration/evaluation partition under strict fingerprint atomicity as a substantive dataset/protocol limitation and redesign the threshold/evaluation strategy instead of iterating partitions.


In [8]:

manifest={
    "dataset":DATASET_NAME,
    "protocol_version":PROTOCOL_VERSION,
    "stage":OUTPUT_STAGE,
    "model_branch_version":MODEL_BRANCH_VERSION,
    "created_at_utc":datetime.now(timezone.utc).isoformat(),
    "partition_membership_sha256":MEMBERSHIP_SHA256,
    "pure_benign_fingerprint_class":PURE_BENIGN_CLASS,
    "source_column":source_col,
    "size_strata":SIZE_BAND_ORDER,
    "pure_benign_rows":int(len(pure)),
    "pure_benign_fingerprints":int(len(group)),
    "max_pure_benign_fingerprint_group_size":int(group["group_size"].max()),
    "fingerprints_touching_multiple_sources":int((group["n_sources_touched"]>1).sum()),
    "tightest_role":tightest_role,
    "tightest_role_pure_benign_rows":tight_target_rows,
    "multiplicity_only_independent_bin_tv_lower_bound":float(tv_lower_bound),
    "independent_bin_total_row_mismatch":int(row_mismatch),
    "attack_scores_accessed":False,
    "attack_outcomes_accessed":False,
    "final_test_outcomes_accessed":False,
    "formal_model_selection_performed":False,
    "formal_notebook01_authorized":False,
    "pre01_membership_modified":False,
    "pre02_modified":False,
    "automatic_repartition_authorized":False,
    "next_authorized_action":"Interpret 00E. If multiplicity balance is arithmetically feasible, freeze PRE-01 v3 source×multiplicity priorities and tolerances before constructing any new membership. If infeasible, do not repartition blindly.",
}
manifest["diagnostic_manifest_hash"]=sha256_bytes(canonical_json_bytes({
    k:v for k,v in manifest.items() if k not in {"created_at_utc","diagnostic_manifest_hash"}
}))
atomic_write_json(OUTPUT_ROOT/"diagnostic_manifest.json",manifest)

print("\n00E COMPLETE")
print("Formal Notebook 01 authorization: BLOCKED")
print("PRE-01 modified: NO")
print("PRE-02 modified: NO")
print("\nReturn:")
for f in [
    "global_pure_benign_multiplicity_distribution.csv",
    "pure_benign_multiplicity_by_role.csv",
    "current_role_multiplicity_tv.csv",
    "global_source_x_multiplicity_cells.csv",
    "current_role_source_x_multiplicity_tv.csv",
    "tightest_role_multiplicity_atomic_lower_bound_by_band.csv",
    "tightest_role_multiplicity_atomic_lower_bound_summary.csv",
    "large_band_source_cell_feasibility.csv",
    "diagnostic_manifest.json",
]:
    print(" -",f)



00E COMPLETE
Formal Notebook 01 authorization: BLOCKED
PRE-01 modified: NO
PRE-02 modified: NO

Return:
 - global_pure_benign_multiplicity_distribution.csv
 - pure_benign_multiplicity_by_role.csv
 - current_role_multiplicity_tv.csv
 - global_source_x_multiplicity_cells.csv
 - current_role_source_x_multiplicity_tv.csv
 - tightest_role_multiplicity_atomic_lower_bound_by_band.csv
 - tightest_role_multiplicity_atomic_lower_bound_summary.csv
 - large_band_source_cell_feasibility.csv
 - diagnostic_manifest.json
